# Frequency-Domain Response of STD Sensitivity Functions

Generates Figs. S2 and S3. This notebook compares analytical linear-response predictions with numerical simulations for the frequency-domain response of the STD sensitivity functions and the effective presynaptic terms.


In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy import signal

from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIGURE_DIR = ROOT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Shared plotting style for manuscript PDFs.
config = {
    "font.family": "sans-serif",
    "font.size": 18.0,
    "axes.titlelocation": "left",
    "axes.titlesize": 19.0,
    "axes.labelsize": 19.0,
    "xtick.labelsize": 17.0,
    "ytick.labelsize": 17.0,
    "legend.fontsize": 17.0,
    "figure.titlesize": 19.0,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.5,
    "lines.markersize": 4.0,
    "patch.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "axes.xmargin": 0.01,
    "axes.ymargin": 0.05,
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.minor.size": 2.0,
    "ytick.minor.size": 2.0,
    "xtick.minor.width": 0.6,
    "ytick.minor.width": 0.6,
    "legend.frameon": False,
    "legend.fancybox": False,
    "image.interpolation": "none",
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
}
plt.rcParams.update(config)


In [ ]:
params = {
    'freq': 1.0,
    'amp': 10.0,
    'tau_d': 0.5,  # Depression time constant (s)
    'beta' : 2.0,  # Steepness of activation function
    'g_M' : 10.0, # Maximum firing rate (Hz) for sigmoid activation.
    'u_c' : 1.0,    # Activation threshold
    'tau_s' : 0.01,  # Synaptic time constant (s),
    'U_init' : 0.15,
    'w0_init' : 1.0,
    'activation' : 'exp',
}


In [ ]:

# -----------------------------
# Parameters (edit as needed)
# -----------------------------
tau_d = params['tau_d']        # depression time constant
U = params['U_init']        # effective utilization parameter

nu0   = 10.0       # baseline presynaptic rate
delta_nu = 0.1       # small modulation amplitude

# Rate constant used in the figure caption.
kappa = 1.0 / tau_d + nu0 * U

# steady-state values for constant nu0
f_w0_star = 1.0 / (1.0 + tau_d * nu0 * U)
f_U_star  = 1.0 / (1.0 + tau_d * nu0 * U)**2

# -----------------------------
# Linear-response transfer functions
# -----------------------------
def H_w0(omega):
    """Complex gain H_{w0}(omega)."""
    return -U * f_w0_star / (kappa + 1j * omega)

def H_U(omega):
    """Complex gain H_U(omega)."""
    return (
        -U * (f_U_star + f_w0_star) / (kappa + 1j * omega)
        + U**2 * nu0 * f_w0_star / (kappa + 1j * omega)**2
    )
def H_C_pre_w0(omega):
    """Complex gain H_{f_w0}(omega)."""
    return f_w0_star + H_w0(omega) * nu0

def H_C_pre_U(omega):
    """Complex gain H_{f_U}(omega)."""
    return f_U_star + H_U(omega) * nu0

# -----------------------------
# Full sensitivity dynamics for simulation
#   (consistent with the linearization used in the text)
#   df^{w0}/dt = 1/tau_d - (1/tau_d + nu(t)*U) f^{w0}
#   df^U/dt   = 1/tau_d - (1/tau_d + nu(t)*U) f^U - nu(t)*U f^{w0}
# -----------------------------
def stp_sensitivity_ode(t, y, tau_d, nu0, delta_nu, omega, U):
    f_w0, f_U = y
    nu_t = nu0 + delta_nu * np.cos(omega * t)

    df_w0 = 1.0 / tau_d - (1.0 / tau_d + nu_t * U) * f_w0
    df_U  = 1.0 / tau_d - (1.0 / tau_d + nu_t * U) * f_U - nu_t * U * f_w0

    return [df_w0, df_U]

def estimate_gain_phase(omega,
                        tau_d=tau_d, nu0=nu0,
                        delta_nu=delta_nu, U=U,
                        n_cycles=100, points_per_cycle=200):
    """
    """
    T = 2.0 * np.pi / omega
    t_end = n_cycles * T

    t_eval = np.linspace(0.0, t_end, int(n_cycles * points_per_cycle))

    y0 = [f_w0_star, f_U_star]

    sol = solve_ivp(
        stp_sensitivity_ode,
        t_span=(0.0, t_end),
        y0=y0,
        t_eval=t_eval,
        args=(tau_d, nu0, delta_nu, omega, U),
        rtol=1e-7,
        atol=1e-9,
    )

    t = sol.t
    f_w0 = sol.y[0]
    f_U  = sol.y[1]

    nu_t = nu0 + delta_nu * np.cos(omega * t)
    delta_nu_t = nu_t - np.mean(nu_t)

    C_pre_w0 = f_w0 * nu_t
    C_pre_U  = f_U  * nu_t

    delta_f_w0 = f_w0 - np.mean(f_w0)
    delta_f_U  = f_U  - np.mean(f_U)
    delta_C_pre_w0 = C_pre_w0 - np.mean(C_pre_w0)
    delta_C_pre_U  = C_pre_U  - np.mean(C_pre_U)

    half = len(t) // 2
    t = t[half:]
    delta_nu_t = delta_nu_t[half:]
    delta_f_w0 = delta_f_w0[half:]
    delta_f_U  = delta_f_U[half:]
    delta_C_pre_w0 = delta_C_pre_w0[half:]
    delta_C_pre_U  = delta_C_pre_U[half:]

    # H(ω) ≈ ∫ f(t) e^{-iωt} dt / ∫ δν(t) e^{-iωt} dt
    dt = t[1] - t[0]
    exp_factor = np.exp(-1j * omega * t)

    F_in  = np.sum(delta_nu_t * exp_factor) * dt
    F_w0  = np.sum(delta_f_w0 * exp_factor) * dt
    F_U   = np.sum(delta_f_U  * exp_factor) * dt
    F_C_pre_w0 = np.sum(delta_C_pre_w0 * exp_factor) * dt
    F_C_pre_U  = np.sum(delta_C_pre_U  * exp_factor) * dt

    H_w0_sim = F_w0 / F_in
    H_U_sim  = F_U  / F_in
    H_C_pre_w0_sim = F_C_pre_w0 / F_in
    H_C_pre_U_sim  = F_C_pre_U  / F_in

    return H_w0_sim, H_U_sim, H_C_pre_w0_sim, H_C_pre_U_sim

# -----------------------------
# Frequency grid
# -----------------------------
# Wide frequency range normalized by kappa.
omega_vals = np.logspace(-2, 2, 40) * kappa

# Analytic transfer functions.
Hw0_vals = H_w0(omega_vals)
HU_vals  = H_U(omega_vals)

H_C_pre_U_vals = H_C_pre_U(omega_vals)
H_C_pre_w0_vals = H_C_pre_w0(omega_vals)

gain_w0_th = np.abs(Hw0_vals)
gain_U_th  = np.abs(HU_vals)

gain_C_pre_w0_th = np.abs(H_C_pre_w0_vals)
gain_C_pre_U_th  = np.abs(H_C_pre_U_vals)

phase_w0_th = np.angle(Hw0_vals)
phase_U_th  = np.angle(HU_vals)

phase_C_pre_w0_th = np.angle(H_C_pre_w0_vals)
phase_C_pre_U_th  = np.angle(H_C_pre_U_vals)

# Numerical estimates on a coarser grid.
omega_sim = np.logspace(-2, 2, 15) * kappa

gain_w0_sim = []
gain_U_sim  = []
phase_w0_sim = []
phase_U_sim  = []
gain_C_pre_w0_sim = []
gain_C_pre_U_sim = []
phase_C_pre_w0_sim = []
phase_C_pre_U_sim = []

for w in omega_sim:
    H_w0_sim, H_U_sim, H_C_pre_w0_sim, H_C_pre_U_sim = estimate_gain_phase(w)
    gain_w0_sim.append(np.abs(H_w0_sim))
    gain_U_sim.append(np.abs(H_U_sim))
    phase_w0_sim.append(np.angle(H_w0_sim))
    phase_U_sim.append(np.angle(H_U_sim))
    gain_C_pre_w0_sim.append(np.abs(H_C_pre_w0_sim))
    gain_C_pre_U_sim.append(np.abs(H_C_pre_U_sim))
    phase_C_pre_w0_sim.append(np.angle(H_C_pre_w0_sim))
    phase_C_pre_U_sim.append(np.angle(H_C_pre_U_sim))
gain_w0_sim = np.array(gain_w0_sim)
gain_U_sim  = np.array(gain_U_sim)
phase_w0_sim = np.array(phase_w0_sim)
phase_U_sim  = np.array(phase_U_sim)
gain_C_pre_w0_sim = np.array(gain_C_pre_w0_sim)
gain_C_pre_U_sim = np.array(gain_C_pre_U_sim)
phase_C_pre_w0_sim = np.array(phase_C_pre_w0_sim)
phase_C_pre_U_sim = np.array(phase_C_pre_U_sim)


In [ ]:

# -----------------------------
# Plot
# -----------------------------
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax_gain = axes[0]
ax_phase = axes[1]

# (A) Gain
line_w0, = ax_gain.loglog(omega_vals, gain_w0_th, label=r'$w_0$', lw=2, color='C0')
line_U,  = ax_gain.loglog(omega_vals, gain_U_th,  label=r'$U$',   lw=2, color='C1')

ax_gain.loglog(omega_sim, gain_w0_sim, 'o', ms=4, color='C0')
ax_gain.loglog(omega_sim, gain_U_sim,  's', ms=4, color='C1')

ax_gain.set_ylabel('Gain ' + r'$|H_{Z}^f(\omega)|$')
ax_gain.grid(True, which='both', ls=':')

# (B) Phase
ax_phase.semilogx(omega_vals, phase_w0_th, lw=2, color='C0')
ax_phase.semilogx(omega_vals, phase_U_th,  lw=2, color='C1')

ax_phase.semilogx(omega_sim, phase_w0_sim, 'o', ms=4, color='C0')
ax_phase.semilogx(omega_sim, phase_U_sim,  's', ms=4, color='C1')

ax_phase.set_xlabel(r'Frequency $\omega$')
ax_phase.set_ylabel(r'Phase $\phi_Z^f (\omega) $[rad]')
ax_phase.grid(True, which='both', ls=':')

fig.legend([line_w0, line_U], [r'$w_0$', r'$U$'], loc='center left', bbox_to_anchor=(1.01, 0.5))
plt.tight_layout(rect=[0, 0, 0.97, 1])
plt.savefig(FIGURE_DIR / 'STP-phase-response-f.pdf', bbox_inches='tight')
plt.show()


fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax_gain = axes[0]
ax_phase = axes[1]
# (A) Gain
line_w0, = ax_gain.loglog(omega_vals, gain_C_pre_w0_th, label=r'$w_0$', lw=2, color='C0')
line_U,  = ax_gain.loglog(omega_vals, gain_C_pre_U_th,  label=r'$U$',   lw=2, color='C1')
ax_gain.loglog(omega_sim, gain_C_pre_w0_sim, 'o', ms=4, color='C0')
ax_gain.loglog(omega_sim, gain_C_pre_U_sim,  's', ms=4, color='C1')

ax_gain.set_ylabel(r'Gain $|H_Z^C (\omega)|$')
ax_gain.set_ylim(0.04, 1.1)
ax_gain.grid(True, which='both', ls=':')
# (B) Phase
ax_phase.semilogx(omega_vals, phase_C_pre_w0_th, lw=2, color='C0')
ax_phase.semilogx(omega_vals, phase_C_pre_U_th,  lw=2, color='C1')
ax_phase.semilogx(omega_sim, phase_C_pre_w0_sim, 'o', ms=4, color='C0')
ax_phase.semilogx(omega_sim, phase_C_pre_U_sim,  's', ms=4, color='C1')
ax_phase.set_xlabel(r'Frequency $\omega$')
ax_phase.set_ylabel(r'Phase $\phi_Z^C(\omega)$ [rad]')
ax_phase.grid(True, which='both', ls=':')
fig.legend([line_w0, line_U], [r'$w_0$', r'$U$'], loc='center left', bbox_to_anchor=(1.01, 0.5))
plt.tight_layout(rect=[0, 0, 0.97, 1])
plt.savefig(FIGURE_DIR / 'STP-phase-response-C.pdf', bbox_inches='tight')
plt.show()
